# Aegis Wave - Environment Diagnostic
## Goal: Isolate if TensorFlow itself is hanging or if it is a data/file issue.

Run these cells in order to identify the bottleneck on your M4 MacBook Pro.

In [9]:
import os
import sys
import tensorflow as tf
import numpy as np
import time

print(f"Python Version: {sys.version}")
print(f"TensorFlow Version: {tf.__version__}")
print(f"Platform: {sys.platform}")
print(f"Physical Devices: {tf.config.list_physical_devices()}")
print(f"Eager Execution: {tf.executing_eagerly()}")

Python Version: 3.12.0 (tags/v3.12.0:0fb18b0, Oct  2 2023, 13:03:39) [MSC v.1935 64 bit (AMD64)]
TensorFlow Version: 2.20.0
Platform: win32
Physical Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
Eager Execution: True


In [10]:
# Force CPU to eliminate Metal/GPU conflicts
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
tf.config.set_visible_devices([], "GPU")

print("Generating synthetic data in memory...")
X_synth = np.random.rand(1000, 10, 10).astype("float32")
y_synth = np.random.randint(0, 2, 1000)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(10, 10)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(2, activation="softmax")
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

print("Starting synthetic fit...")
start_time = time.time()
model.fit(X_synth, y_synth, epochs=5, batch_size=32, verbose=2)
print(f"SYNTHETIC TEST SUCCESSFUL! Time taken: {time.time() - start_time:.2f}s")

Generating synthetic data in memory...
Starting synthetic fit...
Epoch 1/5
32/32 - 0s - 7ms/step - accuracy: 0.5000 - loss: 0.7128
Epoch 2/5
32/32 - 0s - 709us/step - accuracy: 0.5210 - loss: 0.6994
Epoch 3/5
32/32 - 0s - 710us/step - accuracy: 0.5290 - loss: 0.6920
Epoch 4/5
32/32 - 0s - 634us/step - accuracy: 0.5410 - loss: 0.6868
Epoch 5/5
32/32 - 0s - 627us/step - accuracy: 0.5610 - loss: 0.6807
SYNTHETIC TEST SUCCESSFUL! Time taken: 0.32s


In [11]:
print("Running matrix multiplication test...")
a = tf.random.normal([2000, 2000])
b = tf.random.normal([2000, 2000])
start = time.time()
c = tf.matmul(a, b)
print(f"Matrix Multiply (2000x2000) took: {time.time() - start:.4f}s")

Running matrix multiplication test...
Matrix Multiply (2000x2000) took: 0.0263s
